In [1]:
import sys; print(sys.executable) 

/home/cjen/mySageMaker/ML/spark-nlp/redaction/dat/.venv/bin/python


In [2]:
import boto3

region = boto3.Session().region_name
s3 = boto3.client("s3", region_name=region)

In [3]:
from sagemaker import Session, s3

session = Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml


In [4]:
import boto3

boto_session = boto3.Session()

region = boto_session.region_name
account = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account}"

inputs = s3.S3Uploader.upload("data", "s3://{}/spark-nlp/redaction/jgh-msss-prem/redacted_input".format(bucket))

print("S3 location " + inputs)

S3 location s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_input


In [5]:
import subprocess, sys, os

java_check = subprocess.run(["java", "-version"], capture_output=True)
if java_check.returncode != 0:
    print("Java not found — installing default-jdk ...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "default-jdk"], check=True)
    print("Java installed.")
else:
    print("Java already available:", java_check.stderr.decode().split('\n')[0])

java_home_candidates = [
    "/usr/lib/jvm/java-11-openjdk-amd64",
    "/usr/lib/jvm/java-11-amazon-corretto",
    "/usr/lib/jvm/java-8-openjdk-amd64",
    "/usr/lib/jvm/java-8-amazon-corretto",
]
for candidate in java_home_candidates:
    if os.path.isdir(candidate):
        os.environ["JAVA_HOME"] = candidate
        break
else:
    result = subprocess.run(
        ["java", "-XshowSettings:all", "-version"],
        capture_output=True, text=True
    )
    for line in result.stderr.splitlines():
        if "java.home" in line:
            java_home = line.split("=")[1].strip()
            if java_home.endswith("/jre"):
                java_home = java_home[:-4]
            os.environ["JAVA_HOME"] = java_home
            break

print("JAVA_HOME:", os.environ.get("JAVA_HOME", "NOT SET"))

Java already available: openjdk version "11.0.30" 2026-01-20
JAVA_HOME: /usr/lib/jvm/java-11-openjdk-amd64


In [6]:
!pip install -q \
    "sagemaker>=2.0,<3.0" \
    "pyspark>=3.3,<3.6" \
    "spark-nlp==5.5.3" \
    "spark-nlp-display" \
    "boto3"

print("All packages installed.")

All packages installed.


In [7]:
import sagemaker
from sagemaker import get_execution_role

sm_session   = sagemaker.Session(boto_session=boto_session)

try:
    role = get_execution_role(sagemaker_session=sm_session)
except Exception:
    role = os.environ.get("SAGEMAKER_ROLE", "<YOUR_SAGEMAKER_EXECUTION_ROLE_ARN>")
    print("WARNING: Could not auto-detect execution role. Set SAGEMAKER_ROLE env var.")

region = boto_session.region_name or sm_session.boto_region_name
bucket = sm_session.default_bucket()
prefix = "spark-nlp/redaction"

print(f"SageMaker SDK version : {sagemaker.__version__}")
print(f"Region                : {region}")
print(f"Default S3 bucket     : {bucket}")
print(f"Execution role ARN    : {role}")

SageMaker SDK version : 2.257.1
Region                : us-west-2
Default S3 bucket     : sagemaker-us-west-2-493644444178
Execution role ARN    : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd


In [8]:
# ── Verify this notebook is running within a SageMaker environment ─────────────
import os, json, urllib.request

print("=" * 60)
print("  SageMaker Session Proof")
print("=" * 60)

# 1. Active SageMaker SDK session object
print(f"\n[SDK] sagemaker.__version__  : {sagemaker.__version__}")
print(f"[SDK] Session boto region    : {sm_session.boto_region_name}")
print(f"[SDK] Default S3 bucket      : {sm_session.default_bucket()}")
print(f"[SDK] Execution role ARN     : {role}")
print(f"[SDK] Boto3 caller identity  : {boto3.client('sts').get_caller_identity()['Arn']}")

# 2. SageMaker-specific env vars (set automatically by SageMaker runtimes)
sm_keys = sorted(k for k in os.environ if k.startswith("SM_") or "SAGEMAKER" in k.upper())
if sm_keys:
    print(f"\n[ENV] SageMaker env vars found ({len(sm_keys)}):")
    for k in sm_keys:
        print(f"      {k} = {os.environ[k]}")
else:
    print("\n[ENV] No SM_*/SAGEMAKER_* env vars — typical for Studio JupyterLab or local run.")

# 3. /opt/ml/metadata/resource-metadata.json
#    Present in Studio domains, Processing jobs, and Training jobs.
meta_path = "/opt/ml/metadata/resource-metadata.json"
if os.path.exists(meta_path):
    with open(meta_path) as fh:
        meta = json.load(fh)
    print(f"\n[META] {meta_path}:")
    print(json.dumps(meta, indent=4))
else:
    print(f"\n[META] {meta_path} not found (not inside a managed job container).")

# 4. EC2 IMDSv2 endpoint — reachable on SageMaker Notebook Instances and job instances.
#    If this prints an instance type, the notebook is running on a real SageMaker host.
_imds = "http://169.254.169.254/latest"
try:
    token = urllib.request.urlopen(
        urllib.request.Request(
            _imds + "/api/token",
            headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
            method="PUT",
        ), timeout=1
    ).read().decode()
    inst_type = urllib.request.urlopen(
        urllib.request.Request(
            _imds + "/meta-data/instance-type",
            headers={"X-aws-ec2-metadata-token": token},
        ), timeout=1
    ).read().decode()
    print(f"\n[IMDS] EC2 instance type : {inst_type}  ← confirms managed SageMaker host")
except Exception:
    print("\n[IMDS] EC2 IMDS not reachable — expected when running locally.")
    print("       On a real SageMaker Notebook Instance the instance type would appear here.")

print("\n✓ SageMaker SDK session is authenticated and active.")

  SageMaker Session Proof

[SDK] sagemaker.__version__  : 2.257.1
[SDK] Session boto region    : us-west-2
[SDK] Default S3 bucket      : sagemaker-us-west-2-493644444178
[SDK] Execution role ARN     : arn:aws:iam::493644444178:role/aws-reserved/sso.amazonaws.com/ap-southeast-1/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd
[SDK] Boto3 caller identity  : arn:aws:sts::493644444178:assumed-role/AWSReservedSSO_AWSAdministratorAccess_e93147afb9349dfd/mindy@theclinician.com

[ENV] No SM_*/SAGEMAKER_* env vars — typical for Studio JupyterLab or local run.

[META] /opt/ml/metadata/resource-metadata.json not found (not inside a managed job container).

[IMDS] EC2 IMDS not reachable — expected when running locally.
       On a real SageMaker Notebook Instance the instance type would appear here.

✓ SageMaker SDK session is authenticated and active.


In [9]:
import sys
print("Python executable:", sys.executable)

import sparknlp
from sparknlp.base import *
from sparknlp.annotator import *
from sparknlp.pretrained import PretrainedPipeline

import pyspark.sql.functions as F
from pyspark.sql.types import StringType, ArrayType

import pandas as pd
import re
import warnings

warnings.filterwarnings("ignore")
print("Spark NLP version:", sparknlp.version())

Python executable: /home/cjen/mySageMaker/ML/spark-nlp/redaction/dat/.venv/bin/python
Spark NLP version: 5.5.3


In [10]:
params = {
    "spark.driver.memory"           : "16G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize"    : "2000M",
    "spark.ui.enabled"              : "false",
}

spark = sparknlp.start(params=params)
spark

26/03/26 08:08:59 WARN Utils: Your hostname, MindyJen1008 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/26 08:09:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/cjen/mySageMaker/ML/customer_sentiment/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/cjen/.ivy2/cache
The jars for the packages stored in: /home/cjen/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6de060c9-bab4-411e-861e-b0a19987e56c;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;5.5.3 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	f

## Run Spark-NLP on SageMaker via PySparkProcessor

The cells below use **`sagemaker.spark.processing.PySparkProcessor`** (SageMaker SDK v2.x) to:

1. **Write** `redaction_script.py` — the PySpark + Spark-NLP script that runs *inside* the managed container
2. **Submit** a Processing Job — SageMaker provisions the instance, mounts the S3 input, runs the script, and copies the output back to S3 automatically
3. **Preview** the results directly from S3 — the raw clinical data never touches the notebook's local disk

Data flow: `S3 input` → *(SageMaker mounts)* → `container /opt/ml/processing/input` → Spark-NLP pipeline → `container /opt/ml/processing/output` → *(SageMaker copies back)* → `S3 output`

In [11]:
%%writefile redaction_script.py
"""
Spark-NLP de-identification processing script.
Executed inside a SageMaker PySparkProcessor container via spark-submit.
Models are pre-staged on S3 and mounted into /opt/ml/processing/models/.
"""
import os, sys, subprocess, glob, traceback, zipfile

# ── Early diagnostics (always visible in CloudWatch) ─────────────────────────
print("=" * 60, flush=True)
print(f"Python  : {sys.version}", flush=True)
print(f"CWD     : {os.getcwd()}", flush=True)
print(f"PATH    : {os.environ.get('PATH','')}", flush=True)
print("=" * 60, flush=True)

# ── Install Python-side bindings (JVM JAR is loaded via --jars) ───────────────
pip_result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "spark-nlp==5.5.3", "openpyxl", "pandas<2.0"],
    capture_output=True, text=True,
)
print(f"pip exit={pip_result.returncode}", flush=True)
if pip_result.returncode != 0:
    print("pip stdout:", pip_result.stdout[-3000:], flush=True)
    print("pip stderr:", pip_result.stderr[-3000:], flush=True)

try:
    import sparknlp
    from sparknlp.base import DocumentAssembler
    from sparknlp.annotator import Tokenizer, WordEmbeddingsModel, NerDLModel, NerConverter
    from pyspark.ml import Pipeline
    from pyspark.sql import SparkSession
    import pyspark.sql.functions as F
    import pandas as pd
    print(f"Imports OK — spark-nlp {sparknlp.version()}", flush=True)
    print(f"pandas  {pd.__version__}", flush=True)
except Exception:
    print("IMPORT ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

INPUT_DIR  = "/opt/ml/processing/input"
MODELS_DIR = "/opt/ml/processing/models"
OUTPUT_DIR = "/opt/ml/processing/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Unzip pre-staged models ───────────────────────────────────────────────────
LOCAL_MODELS = "/tmp/sparknlp_models"
os.makedirs(LOCAL_MODELS, exist_ok=True)

# SageMaker mounts each ProcessingInput source into a directory;
# the zip file lands inside that directory (not at the path itself).
for model_name, mount_subdir in [("glove_100d", "glove"), ("ner_dl", "ner")]:
    mount_dir = os.path.join(MODELS_DIR, mount_subdir)
    zips = glob.glob(os.path.join(mount_dir, "*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip found in {mount_dir}; contents: {os.listdir(mount_dir)}")
    zip_path  = zips[0]
    dest_path = os.path.join(LOCAL_MODELS, model_name)
    if not os.path.exists(dest_path):
        print(f"Unzipping {zip_path} → {dest_path}", flush=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(dest_path)
    print(f"  {model_name}: {os.listdir(dest_path)[:3]}", flush=True)

GLOVE_PATH = "file://" + os.path.join(LOCAL_MODELS, "glove_100d")
NER_PATH   = "file://" + os.path.join(LOCAL_MODELS, "ner_dl")

try:
    spark = SparkSession.builder.appName("SparkNLP-Redaction").getOrCreate()
    print(f"Spark {spark.version} started", flush=True)
except Exception:
    print("SPARKSESSION ERROR:", flush=True)
    traceback.print_exc()
    sys.exit(1)

# ── Read input ────────────────────────────────────────────────────────────────
xlsx_files = (
    glob.glob(os.path.join(INPUT_DIR, "**/*.xlsx"), recursive=True)
    + glob.glob(os.path.join(INPUT_DIR, "*.xlsx"))
)
if not xlsx_files:
    raise FileNotFoundError(f"No .xlsx files found under {INPUT_DIR}")

print(f"Reading: {xlsx_files[0]}", flush=True)
pdf = pd.read_excel(xlsx_files[0], dtype=str).fillna("")
sdf = spark.createDataFrame(pdf)
print(f"Input  : {sdf.count()} rows × {len(sdf.columns)} columns", flush=True)

print("\nActual DataFrame columns:")
for idx, c in enumerate(sdf.columns):
    print(f"  [{idx:02d}] {repr(c)}")

# ── Locate target columns ─────────────────────────────────────────────────────
COL_ASPECTS     = "Aspects positifs de l'expérience globale"
COL_SUGGESTIONS = "Suggestions d'amélioration de l'expérience globale"

def find_column(df, target):
    if target in df.columns:
        return target
    for c in df.columns:
        if c.strip() == target.strip():
            print(f"  WARNING: matched '{c}' for target '{target}' after stripping")
            return c
    return None

actual_aspects     = find_column(sdf, COL_ASPECTS)
actual_suggestions = find_column(sdf, COL_SUGGESTIONS)

if actual_aspects is None:
    raise ValueError(f"Column not found: {repr(COL_ASPECTS)}\nAvailable: {sdf.columns}")
if actual_suggestions is None:
    raise ValueError(f"Column not found: {repr(COL_SUGGESTIONS)}\nAvailable: {sdf.columns}")

print(f"aspects     → {repr(actual_aspects)}")
print(f"suggestions → {repr(actual_suggestions)}")

SAFE_ASPECTS     = "_nlp_aspects"
SAFE_SUGGESTIONS = "_nlp_suggestions"

sdf_safe = (
    sdf
    .withColumn(SAFE_ASPECTS,     sdf[actual_aspects])
    .withColumn(SAFE_SUGGESTIONS, sdf[actual_suggestions])
)

# ── NER pipeline ──────────────────────────────────────────────────────────────
TEXT_COLS = {
    SAFE_ASPECTS    : "entities_aspects",
    SAFE_SUGGESTIONS: "entities_suggestions",
}

result_df = sdf_safe
for i, (col_name, entities_col) in enumerate(TEXT_COLS.items()):
    print(f"\n[Pipeline {i+1}/{len(TEXT_COLS)}] NER on '{col_name}'", flush=True)

    doc  = DocumentAssembler().setInputCol(col_name).setOutputCol(f"_doc{i}")
    tok  = Tokenizer().setInputCols([f"_doc{i}"]).setOutputCol(f"_tok{i}")
    emb  = WordEmbeddingsModel.load(GLOVE_PATH) \
               .setInputCols([f"_doc{i}", f"_tok{i}"]).setOutputCol(f"_emb{i}")
    ner  = NerDLModel.load(NER_PATH) \
               .setInputCols([f"_doc{i}", f"_tok{i}", f"_emb{i}"]).setOutputCol(f"_ner{i}")
    conv = NerConverter() \
               .setInputCols([f"_doc{i}", f"_tok{i}", f"_ner{i}"]).setOutputCol(entities_col)

    pipeline  = Pipeline(stages=[doc, tok, emb, ner, conv])
    result_df = (
        pipeline.fit(result_df)
                .transform(result_df)
                .drop(f"_doc{i}", f"_tok{i}", f"_emb{i}", f"_ner{i}")
    )
    n = result_df.count()
    print(f"  Done — {n} rows", flush=True)

result_df = result_df.drop(SAFE_ASPECTS, SAFE_SUGGESTIONS)

# ── Write output ──────────────────────────────────────────────────────────────
out_path = "file://" + os.path.join(OUTPUT_DIR, "redacted")
result_df.coalesce(1).write.mode("overwrite").parquet(out_path)
print(f"\nSaved to: {out_path}", flush=True)

spark.stop()

Overwriting redaction_script.py


In [12]:
# ── Resolve a SageMaker-assumable IAM execution role ──────────────────────────
# SSO roles (aws-reserved/sso.amazonaws.com/...) cannot be assumed by
# sagemaker.amazonaws.com.  This cell finds or creates a dedicated role.
import boto3, json, time

iam          = boto3.client("iam")
SM_ROLE_NAME = "SageMakerExecutionRole-SparkNLP"

TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect"   : "Allow",
        "Principal": {"Service": "sagemaker.amazonaws.com"},
        "Action"   : "sts:AssumeRole",
    }],
})

try:
    sm_role = iam.get_role(RoleName=SM_ROLE_NAME)["Role"]["Arn"]
    print(f"Reusing existing role: {sm_role}")

except iam.exceptions.NoSuchEntityException:
    print(f"Creating '{SM_ROLE_NAME}' ...")
    sm_role = iam.create_role(
        RoleName                 = SM_ROLE_NAME,
        AssumeRolePolicyDocument = TRUST_POLICY,
        Description              = "SageMaker execution role for Spark-NLP processing jobs",
    )["Role"]["Arn"]

    for policy_arn in [
        "arn:aws:iam::aws:policy/AmazonSageMakerFullAccess",
        "arn:aws:iam::aws:policy/AmazonS3FullAccess",
    ]:
        iam.attach_role_policy(RoleName=SM_ROLE_NAME, PolicyArn=policy_arn)
        print(f"  Attached {policy_arn}")

    print("Waiting 15 s for IAM propagation ...")
    time.sleep(15)

print(f"\nsm_role = {sm_role}")

Reusing existing role: arn:aws:iam::493644444178:role/SageMakerExecutionRole-SparkNLP

sm_role = arn:aws:iam::493644444178:role/SageMakerExecutionRole-SparkNLP


In [13]:
# ── Download spark-nlp assembly JAR and stage on S3 ───────────────────────────
import urllib.request, os
from sagemaker.s3 import S3Uploader

NLP_VER  = "5.5.3"
jar_name = f"spark-nlp-assembly-{NLP_VER}.jar"
local    = f"/tmp/{jar_name}"

candidate_urls = [
    f"https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/jars/{jar_name}",
    (
        f"https://repo1.maven.org/maven2/com/johnsnowlabs/nlp/"
        f"spark-nlp-assembly_2.12/{NLP_VER}/"
        f"spark-nlp-assembly_2.12-{NLP_VER}.jar"
    ),
]

downloaded = False
for url in candidate_urls:
    try:
        print(f"Trying  {url}")
        urllib.request.urlretrieve(url, local)
        size_mb = os.path.getsize(local) / 1_048_576
        print(f"  size  {size_mb:.1f} MB")
        downloaded = True
        break
    except Exception as e:
        print(f"  ✗  {e}")

if not downloaded:
    raise RuntimeError(
        "Could not download the spark-nlp assembly JAR.\n"
        f"Download manually from https://github.com/JohnSnowLabs/spark-nlp/releases/tag/{NLP_VER}\n"
        f"and save as /tmp/{jar_name}, then re-run from the S3Uploader line."
    )

# ── Validate it's actually a JAR (ZIP magic bytes start with PK) ──────────────
with open(local, "rb") as f:
    magic = f.read(4)
if magic[:2] != b"PK":
    raise RuntimeError(
        f"Downloaded file is NOT a valid JAR — got header {magic!r} instead of b'PK...'.\n"
        "The URL likely returned an HTML error page. Try the manual download above."
    )
print(f"JAR valid  (magic={magic!r}, size={os.path.getsize(local)/1_048_576:.0f} MB)")

dst_s3_dir      = f"s3://{bucket}/{prefix}/jars"
S3Uploader.upload(local, dst_s3_dir, sagemaker_session=sm_session)
sparknlp_jar_s3 = f"{dst_s3_dir}/{jar_name}"
print(f"Staged at  {sparknlp_jar_s3}")

Trying  https://s3.amazonaws.com/auxdata.johnsnowlabs.com/public/jars/spark-nlp-assembly-5.5.3.jar
  size  605.3 MB
JAR valid  (magic=b'PK\x03\x04', size=605 MB)
Staged at  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jars/spark-nlp-assembly-5.5.3.jar


In [14]:
# ── Download Spark-NLP pretrained models and stage on S3 ──────────────────────
# The SageMaker processing container has no internet access, so we pre-fetch the
# models here (notebook has internet) and upload them to S3 as zip archives.
import os, shutil, glob
from sagemaker.s3 import S3Uploader

MODEL_CACHE = os.path.expanduser("~/cache_pretrained")
os.makedirs(MODEL_CACHE, exist_ok=True)

# Download both models into the default spark-nlp cache
print("Downloading glove_100d ...")
WordEmbeddingsModel.pretrained("glove_100d", "en")
print("Downloading ner_dl ...")
NerDLModel.pretrained("ner_dl", "en")

# Find the downloaded model directories
def find_model_dir(cache, prefix):
    matches = sorted(glob.glob(os.path.join(cache, f"{prefix}_en_*")))
    if not matches:
        raise FileNotFoundError(f"No cached model for prefix '{prefix}' under {cache}")
    return matches[-1]   # newest

glove_dir = find_model_dir(MODEL_CACHE, "glove_100d")
ner_dir   = find_model_dir(MODEL_CACHE, "ner_dl")
print(f"glove dir : {glove_dir}")
print(f"ner dir   : {ner_dir}")

# Zip and upload each model directory
models_s3_prefix = f"s3://{bucket}/{prefix}/models"
for name, model_dir in [("glove_100d", glove_dir), ("ner_dl", ner_dir)]:
    zip_path = f"/tmp/{name}"
    shutil.make_archive(zip_path, "zip", model_dir)
    full_zip = zip_path + ".zip"
    print(f"Uploading {full_zip} ({os.path.getsize(full_zip)/1_048_576:.0f} MB) ...")
    S3Uploader.upload(full_zip, models_s3_prefix, sagemaker_session=sm_session)
    print(f"  → {models_s3_prefix}/{name}.zip")

glove_model_s3 = f"{models_s3_prefix}/glove_100d.zip"
ner_model_s3   = f"{models_s3_prefix}/ner_dl.zip"
print(f"\nModels staged:\n  {glove_model_s3}\n  {ner_model_s3}")

glove_100d download started this may take some time.
Approximate size to download 145.3 MB
[ | ]

26/03/26 08:11:19 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/26 08:11:19 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


glove_100d download started this may take some time.
Approximate size to download 145.3 MB
Download done! Loading the resource.
[OK!]
ner_dl download started this may take some time.
Approximate size to download 13.6 MB
[ | ]

26/03/26 08:11:24 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/26 08:11:24 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
26/03/26 08:11:24 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_dl download started this may take some time.
Approximate size to download 13.6 MB
Download done! Loading the resource.
[ / ]

2026-03-26 08:11:28.435623: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-26 08:11:28.566239: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_nam

[OK!]
glove dir : /home/cjen/cache_pretrained/glove_100d_en_2.4.0_2.4_1579690104032
ner dir   : /home/cjen/cache_pretrained/ner_dl_en_2.4.3_2.4_1584624950746
Uploading /tmp/glove_100d.zip (147 MB) ...
  → s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/models/glove_100d.zip
Uploading /tmp/ner_dl.zip (14 MB) ...
  → s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/models/ner_dl.zip

Models staged:
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/models/glove_100d.zip
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/models/ner_dl.zip


In [15]:
# ── Submit Spark-NLP job to SageMaker ─────────────────────────────────────────
from sagemaker.spark.processing import PySparkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

s3_input_path  = f"s3://{bucket}/spark-nlp/redaction/jgh-msss-prem/redacted_input"
s3_output_path = f"s3://{bucket}/spark-nlp/redaction/jgh-msss-prem/redacted_output"

spark_processor = PySparkProcessor(
    base_job_name          = "spark-nlp-redaction",
    framework_version      = "3.3",
    role                   = sm_role,
    instance_count         = 1,
    instance_type          = "ml.m5.4xlarge",  # 16 vCPU / 64 GB
    max_runtime_in_seconds = 3600,
    sagemaker_session      = sm_session,
)

spark_processor.run(
    submit_app   = "redaction_script.py",
    submit_jars  = [sparknlp_jar_s3],
    configuration = [{
        "Classification": "spark-defaults",
        "Properties": {
            "spark.serializer"                : "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max" : "2000M",
            "spark.driver.memory"             : "16G",
            "spark.driver.maxResultSize"      : "4000M",
            "spark.executor.memory"           : "20G",
            "spark.executor.memoryOverhead"   : "4G",
        },
    }],
    inputs = [
        ProcessingInput(
            source      = s3_input_path,
            destination = "/opt/ml/processing/input",
        ),
        # SageMaker always mounts to a directory; the zip lands inside it
        ProcessingInput(
            source      = glove_model_s3,
            destination = "/opt/ml/processing/models/glove",
            input_name  = "glove-model",
            s3_input_mode = "File",
        ),
        ProcessingInput(
            source      = ner_model_s3,
            destination = "/opt/ml/processing/models/ner",
            input_name  = "ner-model",
            s3_input_mode = "File",
        ),
    ],
    outputs = [
        ProcessingOutput(
            source      = "/opt/ml/processing/output",
            destination = s3_output_path,
        )
    ],
    logs = True,
    wait = True,
)

print(f"\nJob complete. Results at: {s3_output_path}")

INFO:sagemaker:Creating processing-job with name spark-nlp-redaction-2026-03-26-15-12-21-873


............03-26 15:14 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '--jars', 's3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jars/spark-nlp-assembly-5.5.3.jar', '/opt/ml/processing/input/code/redaction_script.py']
03-26 15:14 smspark.cli  INFO     Raw spark options before processing: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-26 15:14 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/redaction_script.py']
03-26 15:14 smspark.cli  INFO     Rendered spark options: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-26 15:14 smspark.cli  INFO     Initializing processing job.
03-26 15:14 smspark-submit INFO     {'current_host': 'algo-1', 'current_instance_type': 'ml.

In [16]:
# ── Fetch ALL CloudWatch logs for the last Processing Job ─────────────────────
import boto3, time

logs      = boto3.client("logs", region_name=region)
job_name  = spark_processor.latest_job.job_name
log_group = "/aws/sagemaker/ProcessingJobs"

print(f"Job      : {job_name}")
print(f"LogGroup : {log_group}\n")

for attempt in range(10):
    resp    = logs.describe_log_streams(logGroupName=log_group, logStreamNamePrefix=job_name)
    streams = resp.get("logStreams", [])
    if streams:
        break
    print(f"Waiting for log streams... ({attempt+1}/10)")
    time.sleep(5)

if not streams:
    print("No log streams found.")
else:
    for stream in streams:
        name = stream["logStreamName"]
        print(f"\n{'='*70}")
        print(f"STREAM: {name}")
        print("=" * 70)

        # Paginate through ALL events
        all_events = []
        kwargs = dict(logGroupName=log_group, logStreamName=name, startFromHead=True)
        while True:
            resp2 = logs.get_log_events(**kwargs)
            events = resp2.get("events", [])
            if not events:
                break
            all_events.extend(events)
            token = resp2.get("nextForwardToken")
            if token == kwargs.get("nextToken"):
                break
            kwargs["nextToken"] = token

        print(f"(total {len(all_events)} log events)")
        for ev in all_events:
            print(ev["message"])

Job      : spark-nlp-redaction-2026-03-26-15-12-21-873
LogGroup : /aws/sagemaker/ProcessingJobs


STREAM: spark-nlp-redaction-2026-03-26-15-12-21-873/algo-1-1774537978
(total 9341 log events)
03-26 15:14 smspark.cli  INFO     Parsing arguments. argv: ['/usr/local/bin/smspark-submit', '--jars', 's3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jars/spark-nlp-assembly-5.5.3.jar', '/opt/ml/processing/input/code/redaction_script.py']
03-26 15:14 smspark.cli  INFO     Raw spark options before processing: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files': None, 'verbose': False}
03-26 15:14 smspark.cli  INFO     App and app arguments: ['/opt/ml/processing/input/code/redaction_script.py']
03-26 15:14 smspark.cli  INFO     Rendered spark options: {'jars': 's3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jars/spark-nlp-assembly-5.5.3.jar', 'class_': None, 'py_files': None, 'files':

In [17]:
# ── Preview results from S3 (no local download of raw input) ──────────────────
import io, boto3, pandas as pd

parsed_base = s3_output_path.replace("s3://", "").split("/", 1)
s3_bucket   = parsed_base[0]
s3_prefix   = parsed_base[1].rstrip("/") + "/"

paginator    = boto3.client("s3").get_paginator("list_objects_v2")
output_files = [
    f"s3://{s3_bucket}/{obj['Key']}"
    for page in paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix)
    for obj in page.get("Contents", [])
]
print(f"Output files ({len(output_files)}):")
for f in output_files:
    print(f"  {f}")

parquet_files = [f for f in output_files if f.endswith(".parquet")]
if parquet_files:
    parsed     = parquet_files[0].replace("s3://", "").split("/", 1)
    obj        = boto3.client("s3").get_object(Bucket=parsed[0], Key=parsed[1])
    preview_df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    print(f"\nOutput preview  ({len(preview_df)} rows × {len(preview_df.columns)} cols):")
    pd.set_option("display.max_colwidth", 120)
    print(preview_df.head(5).to_string(index=False))
else:
    print("\nNo parquet output found yet — run the processor cell first.")


Output files (8):
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted/._SUCCESS.crc
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted/.part-00000-ac220549-400a-4269-9bdb-448f3c2d3c32-c000.snappy.parquet.crc
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted/.part-00000-d0cc1aa8-379f-486a-bf08-dc170c508f3a-c000.snappy.parquet.crc
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted/_SUCCESS
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted/part-00000-ac220549-400a-4269-9bdb-448f3c2d3c32-c000.snappy.parquet
  s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted/part-00000-d0cc1aa8-379f-486a-bf08-dc170c508f3a-c000.snappy.

In [18]:
# ── Download parquet output from S3 and load into local Spark session ─────────
import os, shutil, boto3

local_dir = "/tmp/redacted_output"
if os.path.exists(local_dir):
    shutil.rmtree(local_dir)
os.makedirs(local_dir)

parsed_base = s3_output_path.replace("s3://", "").split("/", 1)
s3_bucket   = parsed_base[0]
s3_prefix   = parsed_base[1].rstrip("/") + "/"

s3_client    = boto3.client("s3")
paginator    = s3_client.get_paginator("list_objects_v2")
all_files    = [
    obj["Key"]
    for page in paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix)
    for obj in page.get("Contents", [])
]
parquet_keys = [k for k in all_files if k.endswith(".parquet")]

print(f"Downloading {len(parquet_keys)} parquet file(s) to {local_dir} ...")
for key in parquet_keys:
    fname = os.path.basename(key)
    s3_client.download_file(s3_bucket, key, f"{local_dir}/{fname}")
    print(f"  {fname}")

result_sdf = spark.read.parquet(local_dir)
# Strip trailing/leading spaces from column names (Excel often adds them)
result_sdf = result_sdf.toDF(*[c.strip() for c in result_sdf.columns])
print(f"\nLoaded  : {result_sdf.count()} rows × {len(result_sdf.columns)} columns")
result_sdf.printSchema()


  part-00000-ac220549-400a-4269-9bdb-448f3c2d3c32-c000.snappy.parquet
  part-00000-d0cc1aa8-379f-486a-bf08-dc170c508f3a-c000.snappy.parquet
  redacted_ana.parquet



Loaded  : 16394 rows × 7 columns
root
 |-- Unnamed: 0: string (nullable = true)
 |-- Date de complétion: string (nullable = true)
 |-- Évaluation: string (nullable = true)
 |-- Aspects positifs de l'expérience globale: string (nullable = true)
 |-- Suggestions d'amélioration de l'expérience globale: string (nullable = true)
 |-- entities_aspects: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- annotatorType: string (nullable = true)
 |    |    |-- begin: integer (nullable = true)
 |    |    |-- end: integer (nullable = true)
 |    |    |-- result: string (nullable = true)
 |    |    |-- metadata: map (nullable = true)
 |    |    |    |-- key: string
 |    |    |    |-- value: string (valueContainsNull = true)
 |    |    |-- embeddings: array (nullable = true)
 |    |    |    |-- element: float (containsNull = true)
 |-- entities_suggestions: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- annotatorType: stri

In [19]:
# ── Filter to English-only rows using langdetect ──────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langdetect"], check=True)

from langdetect import detect, LangDetectException
import pyspark.sql.functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

COL_ASPECTS     = "Aspects positifs de l'expérience globale"
COL_SUGGESTIONS = "Suggestions d'amélioration de l'expérience globale"

@udf(StringType())
def detect_lang(text):
    if not text or len(text.strip()) < 10:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

# A row is English if either survey column is detected as English
english_sdf = (
    result_sdf
    .withColumn("lang_aspects",     detect_lang(F.col(COL_ASPECTS)))
    .withColumn("lang_suggestions", detect_lang(F.col(COL_SUGGESTIONS)))
    .filter(
        (F.col("lang_aspects") == "en") | (F.col("lang_suggestions") == "en")
    )
    .drop("lang_aspects", "lang_suggestions")
)

total   = result_sdf.count()
english = english_sdf.count()
print(f"Total rows   : {total}")
print(f"English rows : {english}  ({english/total*100:.1f}%)")

Total rows   : 16394
English rows : 5742  (35.0%)


In [20]:
# ── Redact both columns (skip ORG), insert redacted cols in-place, upload ─────
import os
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from sagemaker.s3 import S3Uploader

@udf(StringType())
def redact_entities(text, entities):
    """Replace non-ORG entity spans with [LABEL].  ORG tokens are left as-is."""
    if not text or not entities:
        return text
    spans = sorted(
        [
            (e.begin, e.end + 1, (e.metadata.get("entity") or "PHI") if e.metadata else "PHI")
            for e in entities
            if e.result and e.result.strip()
            and (e.metadata.get("entity") or "") != "ORG"   # ← skip ORG
        ],
        key=lambda x: x[0],
        reverse=True,  # replace from end → start to preserve offsets
    )
    result = text
    for begin, end, label in spans:
        result = result[:begin] + f"[{label}]" + result[end:]
    return result

# Apply redaction to both columns
redacted_sdf = (
    english_sdf
    .withColumn("redacted_aspects",
                redact_entities(F.col(COL_ASPECTS),     F.col("entities_aspects")))
    .withColumn("redacted_suggestions",
                redact_entities(F.col(COL_SUGGESTIONS), F.col("entities_suggestions")))
)

# Build column order: insert each redacted column immediately after its source
base_cols = [c for c in result_sdf.columns
             if c not in ("entities_aspects", "entities_suggestions")]
ordered_cols = []
for c in base_cols:
    ordered_cols.append(c)
    if c == COL_ASPECTS:
        ordered_cols.append("redacted_aspects")
    elif c == COL_SUGGESTIONS:
        ordered_cols.append("redacted_suggestions")

final_sdf = redacted_sdf.select(*ordered_cols)
print(f"Final table: {final_sdf.count()} English rows × {len(ordered_cols)} columns\n")
final_sdf.show(truncate=120)

# ── Upload to S3 ──────────────────────────────────────────────────────────────
local_out  = "/tmp/redacted_ana"
os.makedirs(local_out, exist_ok=True)
local_file = f"{local_out}/redacted_ana.parquet"

final_sdf.toPandas().to_parquet(local_file, index=False)
print(f"Written locally: {local_file}")

target_s3 = f"s3://{bucket}/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted_ana"
S3Uploader.upload(local_out, target_s3, sagemaker_session=sm_session)
print(f"Uploaded to    : {target_s3}")

Final table: 5746 English rows × 7 columns



+----------+--------------------------------+----------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------+
|Unnamed: 0|              Date de complétion|Évaluation|                                                                                Aspects positifs de l'expérience globale|                                                                                                        redacted_aspects|                                                                      Suggestions d'amélioration de l'expérience globale|                                        

Written locally: /tmp/redacted_ana/redacted_ana.parquet
Uploaded to    : s3://sagemaker-us-west-2-493644444178/spark-nlp/redaction/jgh-msss-prem/redacted_output/redacted_ana
